In [1]:
import cirq
import numpy as np
from typing import List, Sequence

In [2]:
# Two 9-qubit blocks
BLOCK_A = cirq.LineQubit.range(9)
BLOCK_B = cirq.LineQubit.range(9, 18)

def logical_cz(input_state: cirq.OP_TREE) -> cirq.Circuit:
    """
    Transversal CZ (via H–CNOT–H on the target block) for the rotated surface code
    using the encoding from Fig. 8 of https://arxiv.org/abs/2506.04084.
    Qubit 4 of each 9-qubit block carries the logical input.
    """
    def encode_rotated_surface_block(qs):
        """Encoding circuit for one 9-qubit rotated-surface block (matches the original)."""
        ops = []

        # Hadamards on data subset
        ops += [cirq.H.on_each(qs[i] for i in (1, 2, 6, 8))]

        # Parallel CNOT layers (Moments not strictly needed—Cirq groups parallel ops)
        ops += [
            (cirq.CX(qs[1], qs[0]), cirq.CX(qs[6], qs[3])),
            (cirq.CX(qs[4], qs[5]), cirq.CX(qs[8], qs[7])),
            cirq.CX(qs[1], qs[4]),
            cirq.CX(qs[5], qs[3]),
            cirq.CX(qs[1], qs[3]),
            cirq.CX(qs[2], qs[5]),
            cirq.CX(qs[8], qs[4]),
            cirq.CX(qs[8], qs[5]),
        ]
        return ops

    # Build circuit
    c = cirq.Circuit()

    # Encoding for block A, with the provided input_state inserted (qubit 4 is logical)
    c += encode_rotated_surface_block(BLOCK_A)
    c += input_state

    # Encoding for block B
    c += encode_rotated_surface_block(BLOCK_B)

    # Implement logical CZ by H–CNOT–H on the second block (transversal)
    c += cirq.H.on_each(BLOCK_B)
    c += [cirq.CX(a, b) for a, b in zip(BLOCK_A, BLOCK_B)]
    c += cirq.H.on_each(BLOCK_B)

    # Measure all qubits
    c += cirq.measure(*BLOCK_A, *BLOCK_B, key="m")

    return c


In [3]:
def logical_simulator(physical_circuit, repetitions):
    sim = cirq.Simulator()
    logical_results = []
    for i in range(repetitions):
        results = sim.run(physical_circuit, repetitions=1)
        m = results.measurements['m'][0]
        logical_results.append(sum([m[10], m[13], m[16]]) % 2)
    return logical_results
        

In [4]:
def test_input_1():
    two_qubit_c = logical_cz(cirq.X(BLOCK_A[4]))
    logical_1_result = logical_simulator(two_qubit_c, 10)
    assert all(x == 0 for x in logical_1_result)
    

In [5]:
test_input_1()

In [6]:
def test_input_plus():
    two_qubit_c = logical_cz(cirq.H(BLOCK_A[4]))
    logical_1_result = logical_simulator(two_qubit_c, 10)
    assert all(x == 0 for x in logical_1_result)

In [7]:
test_input_plus()

In [8]:
def test_input_minus():
    two_qubit_c = logical_cz([cirq.H(BLOCK_A[4]), cirq.Z(BLOCK_A[4])])
    logical_1_result = logical_simulator(two_qubit_c, 10)
    assert all(x == 0 for x in logical_1_result)

In [9]:
test_input_minus()

In [10]:
def test_input_i():
    two_qubit_c = logical_cz([cirq.H(BLOCK_A[4]), cirq.S(BLOCK_A[4])])
    logical_1_result = logical_simulator(two_qubit_c, 10)
    assert all(x == 0 for x in logical_1_result)

In [11]:
test_input_i()

In [12]:
def test_input_minus_i():
    two_qubit_c = logical_cz([cirq.H(BLOCK_A[4]), cirq.S(BLOCK_A[4])**-1])
    logical_1_result = logical_simulator(two_qubit_c, 10)
    assert all(x == 0 for x in logical_1_result)

In [13]:
test_input_minus_i()